# Apple Game Rank Reranker Training

Google Drive에 저장한 `rank_data.bin`으로 후보 reranker 모델을 학습하고 ONNX로 export합니다.

로컬에서 먼저 생성한 파일을 Drive의 `MyDrive/apple_game/rank_data.bin`에 업로드하세요.

```bash
go run ./cmd/apple-game -gen-rank 500 -rank-out data/generated/rank_data.bin
```

Colab 메뉴에서 **Runtime > Change runtime type > GPU**를 선택한 뒤 위에서부터 실행합니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, shutil, subprocess, textwrap

GITHUB_URL = "https://github.com/scheveningen361/10_apple_game.git"
REPO_DIR = "/content/10_apple_game"
DRIVE_DIR = "/content/drive/MyDrive/apple_game"
DATA_NAME = "rank_data.bin"

SMALL_PT = "models/model_rank_small_colab.pt"
SMALL_ONNX = "models/model_rank_small_colab.onnx"
MID_PT = "models/model_rank_mid_colab.pt"
MID_ONNX = "models/model_rank_mid_colab.onnx"

os.makedirs(DRIVE_DIR, exist_ok=True)
print("Drive dir:", DRIVE_DIR)

In [ ]:
if os.path.exists(os.path.join(REPO_DIR, ".git")):
    %cd /content/10_apple_game
    !git pull
else:
    %cd /content
    !git clone {GITHUB_URL} {REPO_DIR}

%cd {REPO_DIR}
!git log --oneline -3

In [ ]:
!pip -q install onnx
import torch
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

In [ ]:
src = os.path.join(DRIVE_DIR, DATA_NAME)
dst = os.path.join(REPO_DIR, "data/generated", DATA_NAME)
os.makedirs(os.path.dirname(dst), exist_ok=True)

if not os.path.exists(src):
    raise FileNotFoundError(f"Drive에 데이터가 없습니다: {src}")

if not os.path.exists(dst) or os.path.getsize(dst) != os.path.getsize(src):
    shutil.copy2(src, dst)

print("data:", dst, f"{os.path.getsize(dst)/1e6:.1f} MB")

## Small 모델

로컬에서 실험한 `channels=32, blocks=2` 구성입니다. GPU에서는 먼저 이 설정으로 15 epoch를 확인합니다.

In [ ]:
!python rl/train_rank.py \
  --data data/generated/rank_data.bin \
  --out {SMALL_PT} \
  --export {SMALL_ONNX} \
  --epochs 15 \
  --groups-per-batch 128 \
  --channels 32 \
  --blocks 2

In [ ]:
for path in [SMALL_PT, SMALL_ONNX]:
    shutil.copy2(path, os.path.join(DRIVE_DIR, os.path.basename(path)))
    print("copied:", os.path.join(DRIVE_DIR, os.path.basename(path)))

## Mid 모델

Small 모델보다 큰 실험용 구성입니다. 시간이 부족하면 이 셀은 건너뛰어도 됩니다.

In [ ]:
!python rl/train_rank.py \
  --data data/generated/rank_data.bin \
  --out {MID_PT} \
  --export {MID_ONNX} \
  --epochs 15 \
  --groups-per-batch 96 \
  --channels 48 \
  --blocks 3

In [ ]:
for path in [MID_PT, MID_ONNX]:
    shutil.copy2(path, os.path.join(DRIVE_DIR, os.path.basename(path)))
    print("copied:", os.path.join(DRIVE_DIR, os.path.basename(path)))

## 로컬 평가 명령

Drive에서 ONNX를 로컬 `models/` 폴더로 내려받은 뒤 실행합니다.

```powershell
go run -tags nn ./cmd/apple-game `
  -nn-rerank `
  -n 100 `
  -model models/model_rank_small_colab.onnx `
  -ort-lib runtime/onnxruntime.dll `
  -rerank-weight 0.25 `
  -rerank-k 8
```

Mid 모델 평가:

```powershell
go run -tags nn ./cmd/apple-game `
  -nn-rerank `
  -n 100 `
  -model models/model_rank_mid_colab.onnx `
  -ort-lib runtime/onnxruntime.dll `
  -rerank-weight 0.25 `
  -rerank-k 8
```